In [3]:
!pip install pandas

In [4]:
import pandas as pd

train = pd.read_csv(".data//house-prices-advanced-regression-techniques/train.csv")
print(train.shape)
print(train.dtypes.value_counts())

(1460, 81)
str        43
int64      35
float64     3
Name: count, dtype: int64


## Preprocessing

### Drop ID


In [5]:
features = train.drop(columns=["Id", "SalePrice"])
print(features.shape)

(1460, 79)


### Standardzie

In [6]:
numeric_features = features.select_dtypes(include="number").columns
print(numeric_features)

features[numeric_features] = features[numeric_features].apply(
    lambda x: (x - x.mean()) / x.std()
)

Index(['MSSubClass', 'LotFrontage', 'LotArea', 'OverallQual', 'OverallCond',
       'YearBuilt', 'YearRemodAdd', 'MasVnrArea', 'BsmtFinSF1', 'BsmtFinSF2',
       'BsmtUnfSF', 'TotalBsmtSF', '1stFlrSF', '2ndFlrSF', 'LowQualFinSF',
       'GrLivArea', 'BsmtFullBath', 'BsmtHalfBath', 'FullBath', 'HalfBath',
       'BedroomAbvGr', 'KitchenAbvGr', 'TotRmsAbvGrd', 'Fireplaces',
       'GarageYrBlt', 'GarageCars', 'GarageArea', 'WoodDeckSF', 'OpenPorchSF',
       'EnclosedPorch', '3SsnPorch', 'ScreenPorch', 'PoolArea', 'MiscVal',
       'MoSold', 'YrSold'],
      dtype='str')


### missing values

In [7]:
features[numeric_features] = features[numeric_features].fillna(0)
print(features[numeric_features].isnull().sum())

MSSubClass       0
LotFrontage      0
LotArea          0
OverallQual      0
OverallCond      0
YearBuilt        0
YearRemodAdd     0
MasVnrArea       0
BsmtFinSF1       0
BsmtFinSF2       0
BsmtUnfSF        0
TotalBsmtSF      0
1stFlrSF         0
2ndFlrSF         0
LowQualFinSF     0
GrLivArea        0
BsmtFullBath     0
BsmtHalfBath     0
FullBath         0
HalfBath         0
BedroomAbvGr     0
KitchenAbvGr     0
TotRmsAbvGrd     0
Fireplaces       0
GarageYrBlt      0
GarageCars       0
GarageArea       0
WoodDeckSF       0
OpenPorchSF      0
EnclosedPorch    0
3SsnPorch        0
ScreenPorch      0
PoolArea         0
MiscVal          0
MoSold           0
YrSold           0
dtype: int64


### One hot encode


In [8]:
features = pd.get_dummies(features, dummy_na=True)
print(features.shape)

(1460, 330)


In [9]:
def preprocess(raw_train, raw_val):
    label = 'SalePrice'
    # Gộp train + val để standardize cùng thống kê
    features = pd.concat(
        (raw_train.drop(columns=['Id', label]),
         raw_val.drop(columns=['Id']))
    )
    
    # Standardize numeric
    numeric = features.dtypes[features.dtypes != 'object'].index
    features[numeric] = features[numeric].apply(
        lambda x: (x - x.mean()) / x.std()
    )
    features[numeric] = features[numeric].fillna(0)
    
    # One-hot categorical
    features = pd.get_dummies(features, dummy_na=True)
    
    # Tách lại
    n_train = raw_train.shape[0]
    train = features[:n_train].copy()
    train[label] = raw_train[label]
    val = features[n_train:].copy()
    
    return train, val

In [10]:
def k_fold_data(train_df, k):
    folds = []
    folde_size = len(train_df) // k

    for i in range(k):
        val_idx = range(i * folde_size, (i + 1) * folde_size)
        val_data = train_df.iloc[list(val_idx)]
        train_data = train_df.drop(train_df.index[val_idx])
        folds.append((train_data, val_data))
    return folds


def k_fold_train(model_fn, train_df, k, num_epochs, lr):
    val_losses = []

    for i, (train_data, val_data) in enumerate(k_fold_data(train_df, k)):
        model = model_fn()

        val_loss = evaluate(model, val_data)
        val_losses.append(val_loss)
        print(f"Fold {i}, val loss: {val_loss:.4f}")
    avg = sum(val_losses) / len(val_losses)
    print(f"{k}-fold validation: avg val loss: {avg:.4f}")
    return avg

In [11]:
import torch
from torch import nn
import torch.nn.functional as F

# Model baseline: Linear Regression
def get_linear_model():
    return nn.Sequential(nn.Flatten(), nn.LazyLinear(1))

# Loss: MSE trên log(price)
loss_fn = nn.MSELoss()

# Train 1 fold
def train_fold(model, train_loader, val_loader, lr, wd, epochs):
    optimizer = torch.optim.SGD(model.parameters(), lr=lr, weight_decay=wd)
    
    for epoch in range(epochs):
        model.train()
        for X, y in train_loader:
            pred = model(X)
            loss = loss_fn(pred, y)  # y đã là log(price)
            optimizer.zero_grad()
            loss.backward()
            optimizer.step()
    
    # Evaluate
    model.eval()
    with torch.no_grad():
        val_preds = []
        val_labels = []
        for X, y in val_loader:
            val_preds.append(model(X))
            val_labels.append(y)
        preds = torch.cat(val_preds)
        labels = torch.cat(val_labels)
        rmse = torch.sqrt(loss_fn(preds, labels))
    return rmse.item()